# DLThon2 · 역할 A (문서 · 검색) — 코랩 실행

규약집 PDF → 쪽 단위 조각 · 표 행 문장 → BM25 + bge-m3 하이브리드 검색 → 판본 규칙 → 검색 품질 측정.
V3가 나오면 **맨 아래 'A-7' 절만** 다시 돌리면 됩니다.

| 절 | 업무 | 런타임 |
|---|---|---|
| 0 | 준비 (코드 · PDF · 설치) | CPU도 됨 |
| 1 | A-1~3 추출 · 표 행 · 청킹 + 추출 시험 | CPU |
| 2 | A-4 색인 (벡터는 GPU 권장, T4면 충분) | **GPU** |
| 3 | A-4 · A-6 검색과 판본 규칙 써 보기 | |
| 4 | A-5 검색 품질 | |
| 5 | A-7 V3 반영 · V2↔V3 차이 | |
| 6 | 결과를 드라이브에 저장 | |

**미리 할 것** : 구글 드라이브에 `MyDrive/dlthon2/pdf/` 폴더를 만들고 `rulebook_V2.pdf`를 넣는다. V3가 나오면 같은 폴더에 `rulebook_V3.pdf`.
PDF는 깃에 올리지 않는다(`.gitignore`).

## 0. 준비

In [ ]:
# ── 여기만 고치세요 ─────────────────────────────────────────────
REPO_URL  = ""                                # 예: "https://github.com/<아이디>/dlthon2-rag.git"  (비우면 드라이브의 zip 사용)
DRIVE_DIR = "/content/drive/MyDrive/dlthon2"  # pdf/ 폴더가 있는 곳. 결과도 여기 out/ 에 저장
EMBED     = "BAAI/bge-m3"                     # 의미 검색 모델. "" 이면 BM25만
# ────────────────────────────────────────────────────────────────
import os, subprocess, glob, shutil
from google.colab import drive
print("드라이브 연결 중… 브라우저(또는 VS Code)에 뜬 창에서 계정 선택 → '허용'을 누르세요")
drive.mount("/content/drive")

WORK = "/content/dlthon2-rag"
%cd /content
if os.path.exists(WORK) and not os.path.exists(f"{WORK}/.git"):
    shutil.rmtree(WORK)          # zip 으로 푼 옛 코드는 지우고 새 zip 으로 다시 푼다 (세션을 재시작하지 않아도 최신)
if os.path.exists(f"{WORK}/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "-q"], check=False)
if not os.path.exists(WORK):
    if REPO_URL:
        url = REPO_URL
        try:   # 비공개 저장소면 코랩 왼쪽 '열쇠(보안 비밀)'에 GH_TOKEN 을 넣어 둔다
            from google.colab import userdata
            tok = userdata.get("GH_TOKEN")
            url = REPO_URL.replace("https://", f"https://{tok}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, WORK], check=True)
    else:
        zips = sorted(glob.glob(f"{DRIVE_DIR}/dlthon2-rag*.zip"), key=os.path.getmtime)
        assert zips, f"REPO_URL 이 비었고 {DRIVE_DIR} 에 dlthon2-rag*.zip 도 없습니다"
        print("zip :", os.path.basename(zips[-1]), "(드라이브에서 가장 최근 것)")
        subprocess.run(["unzip", "-q", "-o", zips[-1], "-d", "/content"], check=True)
        if not os.path.exists(WORK):            # zip 안 폴더 이름이 다를 때
            top = [d for d in glob.glob("/content/*") if os.path.exists(f"{d}/rag/extract.py")]
            shutil.move(top[0], WORK)

os.makedirs(f"{WORK}/pdf", exist_ok=True)
for p in glob.glob(f"{DRIVE_DIR}/pdf/*.pdf"):
    shutil.copy(p, f"{WORK}/pdf/")
%cd {WORK}
print(sorted(os.listdir("pdf")))
print("코드 판 :", open("BUILD.txt").read().strip() if os.path.exists("BUILD.txt") else "(BUILD.txt 없음 = 11.17 이전 zip)")
assert os.path.exists("scripts/export_retrieval.py"), "옛날 코드입니다. 드라이브에 새 zip(dlthon2-rag_1118.zip 등)을 올리세요"

In [ ]:
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU 없음 (벡터 만들기는 느림)"

## 1. A-1 · A-2 · A-3 : PDF 추출 · 표 행 문장 · 쪽 단위 조각

- 본문과 표를 따로 뽑는다(표 영역은 본문에서 빼고 `pdfplumber` 표 인식). 머리말 · 꼬리말 제거
- 표의 각 행 → `표 41 가족행사별 공동비 상한 V2, 행사: …, 행사당 상한: 22만 원` (`rows.jsonl`)
- 쪽 하나 = 조각 하나. 판본 · 쪽 · 장 · 조항 · 표 · 참조 · 개정 여부 (`pages.jsonl`)
- 기대값(V2) : 쪽 60 · 표 행 342 · 경고 0건

In [ ]:
!python -m rag.extract --doc V2
!head -12 data/V2/report.txt

**추출 시험** : 쪽 번호(꼬리말 대조) · 조항 · 표 번호 = 쪽 번호 · 칸 안 줄바꿈 복원(표 17 '같은 누수의 출동과 후속부품') · 프롬프트에 들어갈 본문

In [ ]:
!python -m scripts.check_extract --doc V2 --pages 17 41 54 | head -80

## 2. A-4 : 색인

조각 402개(쪽 60 + 표 행 342). BM25는 검색할 때 만들고, 여기서는 벡터(`emb.npy`)만 만든다. T4 기준 bge-m3 첫 다운로드 포함 몇 분.

In [ ]:
if EMBED:
    !python -m rag.index --doc V2 --embed {EMBED}
else:
    !python -m rag.index --doc V2

## 3. 검색 · 판본 규칙 써 보기 (A-4 · A-6)

`[판본]` 줄 : 표 07 항목 → 기준일 종류 → 질문 속 날짜 → 부칙 제1조 시행 구간. 날짜가 없으면 `추가 확인`(되묻기).
쪽 줄의 `[ ]` 안 : 검색 순위 · 표 60 색인어 · 참조 따라가기 · 판본 규칙으로 붙은 이유.

In [ ]:
!python -m rag.search "10월 15일에 시어머니 생신 선물 15만 원과 식사 7만 원을 서로 다른 카드로 결제할 예정이야" --doc V2
print()
!python -m rag.search "어머님 생신 식사와 선물을 서로 다른 카드로 결제했어. 공동비로 처리할 수 있을까?" --doc V2

In [ ]:
# 경과조치 : 환불은 원거래 판본, 전량 취소 후 새 주문은 새 날짜
!python -m rag.version "공용 밥솥을 35만 원에 9월 30일 주문 확정했고 10월 3일에 환불받았어" --doc V2 | grep -E "applied|reason"
!python -m rag.version "9월 28일 주문한 로봇청소기를 전량 취소하고 10월 2일 새 주문으로 확정했어" --doc V2 | grep -E "applied|reason"

## 4. A-5 : 검색 품질

정답 조항 · 표가 있는 쪽이 결과에 들어왔는지. 기대값(V2, BM25만) : 검색만 0.902 · 검색+참조+판본 0.985 · 판본 16/16.
벡터를 더하면 검색만 값이 달라진다(RunPod bge-m3 실측 0.931).
팀 질문 세트가 생기면 `--questions eval/questions_team.jsonl`.

In [ ]:
!python -m eval.eval_retrieval --doc V2

**판본 규칙만 채점 (v2)** : 판본을 헷갈리게 만든 30문항 + 처음 보는 15문항. 기대값 30/30 · 15/15

In [ ]:
!python -m eval.eval_version --doc V2
!python -m eval.eval_version --doc V2 --questions eval/questions_version_holdout.jsonl
!python -m eval.eval_version --doc V2 --questions eval/questions_version_team.jsonl
!python -m eval.eval_version --doc V2 --questions eval/questions_version_dev50.jsonl

## 5. A-7 : V3 반영

V3 PDF를 `DRIVE_DIR/pdf/rulebook_V3.pdf`로 넣고 **0절 두 번째 셀(PDF 복사)부터** 다시 실행한 뒤 아래를 순서대로.

1. 추출 → `report.txt` 경고 확인
2. `add_version` (확인만) : 시행일 · 표 07 항목 · 대조표 쪽 · 구조(표 번호 = 쪽 번호, 머리말 · 꼬리말)를 **자동으로 찾아 보여 줌**
3. 내용이 맞으면 `--write`로 `config/versions.json`에 반영. **새 항목**이 나오면 `items`의 `any`에 키워드를 채움
4. 색인 → 품질 → V2↔V3 차이(`results/diff_V2_V3.md`, C에게 '개정 대응 질문' 재료로)

문서마다 '아는 판본'이 다르다 : 같은 12월 날짜 질문도 `--doc V2`면 V2, `--doc V3`면 V3로 판정한다. 이것이 V2/V3 비교 실험의 축.

In [ ]:
V3_PDF = "pdf/rulebook_V3.pdf"
assert os.path.exists(V3_PDF), "V3 PDF가 아직 없습니다 (드라이브 pdf/ 에 넣고 0절 PDF 복사 셀을 다시 실행)"
!python -m rag.extract --doc V3 --pdf {V3_PDF}
!python -m scripts.add_version --doc V3 --pdf {V3_PDF}

위 `[확인할 것]`이 비었고 시행일 · 항목이 PDF와 맞으면 반영합니다. 시행일을 못 찾았다면 `--start 2026-MM-DD`를 붙입니다.

In [ ]:
!python -m scripts.add_version --doc V3 --pdf {V3_PDF} --write
if EMBED:
    !python -m rag.index --doc V3 --embed {EMBED}
else:
    !python -m rag.index --doc V3
!python -m eval.eval_retrieval --doc V3
!python -m eval.eval_version --doc V3
!python -m scripts.diff_versions --old V2 --new V3 --questions eval/questions_seed.jsonl

## 5-1. B · C에게 넘길 검색 결과 파일 (11.19 C 지시 3-2 · 3-4 · 3-5)
드라이브 `dlthon2/B_1119/`에 `개발문항_150.jsonl` · `개발정답_150.csv` · `잡음문항_90.jsonl`(C 검수본이 오면 그 파일), C의 `형식변형_F01-F30.csv`를 올려 둔다.
- 질문 세트마다 메모 있음 / 없음 두 판 : `results/retrieval_<dev150|noise90|fmt30>[_nobrief]_V2.jsonl`. 고정 조각 4 · 6 · 7 · 53쪽, `id` = 입력의 문항번호
- **출력 첫 줄이 `벡터 검색 켬`인지 확인**(아니면 2절 색인부터). 그 줄을 결과와 함께 B · C에게 보낸다
- B는 `hits` · `brief_text` · `version`만 쓴다(`messages`는 A 형식이라 쓰지 않음). 판본 칸은 `version.label`(팀 표기 : V1 · V2 · 해당 없음 · 미정)
- 공개 100은 **토큰 수만** 잰다(3-4). 질문 · 검색 결과는 보지 않는다

In [ ]:
B1119 = f"{DRIVE_DIR}/B_1119"
sets = [("개발문항_150.jsonl", "dev150"), ("잡음문항_90.jsonl", "noise90"), ("형식변형_F01-F30.csv", "fmt30")]
for fname, name in sets:
    path = f"{B1119}/{fname}"
    if not os.path.exists(path):
        print("없음(건너뜀) :", fname); continue
    !python -m scripts.export_retrieval --questions "{path}" --name {name}
    !python -m scripts.export_retrieval --questions "{path}" --name {name}_nobrief --no-brief

# 3-5 : BM25만 vs BM25 + bge-m3 (개발 150 필수 근거)
!python -m eval.eval_dense --dev "{B1119}/개발정답_150.csv"
# 3-3 확인 : 잡음 (c) 30문항이 원문항보다 나빠지지 않는지 (벡터 켬 기준)
!python -m eval.eval_noise --noise "{B1119}/잡음문항_90.jsonl" --dev "{B1119}/개발정답_150.csv" --variant c
# 3-4 : 가장 긴 공개 5문항의 입력 토큰 수 (숫자만). 모델 이름은 B가 알려 준 것으로 바꾼다
PUB = f"{DRIVE_DIR}/배포용_평가키트_V2/01_공개_배포평가셋_100문항.jsonl"
if os.path.exists(PUB):
    !python -m scripts.count_tokens --questions "{PUB}" --ids PUB-086 PUB-093 PUB-091 PUB-075 PUB-028 --models Qwen/Qwen3-4B-AWQ Qwen/Qwen3-32B-AWQ

## 5-2. bge-m3 진단 (11.19 저녁)
벡터를 켜면 '전체 검색 결과 안에 근거'가 126 → 123으로 3문항 줄었다. 어떤 문항인지 찍고,
로컬(벡터 모델을 못 돌리는 PC)에서 같은 검색을 재현할 수 있게 **질문 벡터**를 `data/V2/qemb.npz`로 저장한다.
실행 뒤 **6절**까지 돌리면 `out/<시각>/data/V2/`에 `emb.npy` · `qemb.npz`가 함께 저장된다. 그 폴더를 A 로컬로 가져온다.
공개 100 파일은 넣지 않는다.

In [ ]:
B1119 = f"{DRIVE_DIR}/B_1119"
!python -m eval.eval_dense --dev "{B1119}/개발정답_150.csv" --show
!python -m scripts.dump_query_emb --questions "{B1119}/개발문항_150.jsonl" "{B1119}/잡음문항_90.jsonl" "{B1119}/형식변형_F01-F30.csv"

## 6. 결과를 드라이브에 저장
`config/versions.json`이 바뀌었으면(V3 반영) 로컬 저장소에 복사해 커밋합니다.

In [ ]:
import time
dst = f"{DRIVE_DIR}/out/{time.strftime('%m%d-%H%M')}"
os.makedirs(dst, exist_ok=True)
for d in ("results", "data", "config"):
    if os.path.exists(d):
        shutil.copytree(d, f"{dst}/{d}", dirs_exist_ok=True)
print("->", dst)